# exp_019 V18 Postproc Ablation

Postprocessing ablation notebook for the working `exp_015d` V18 artifact stack.

This notebook is intentionally **not** a submit notebook.
It replays the fixed V18 stack on cached trusted soundscape rows and then benchmarks a small set of low-risk postprocessing variants.

Goal:

1. preserve the already strong `exp_015d` model stack
2. test cheap postprocess ideas before spending more Kaggle submissions
3. identify whether any small postprocess-only change looks promising enough for a later thin submit path

Important caveat:

- this is a **proxy benchmark on cached labeled full rows**, not a fully honest hidden-test or full OOF benchmark
- results are useful for ranking cheap variants, but they should be interpreted conservatively


In [ ]:
# Cell 0 — Input hints and run config
COMPETITION_HINT = None          # e.g. "birdclef-2026"
PERCH_CACHE_HINT = None          # e.g. "perch-meta"
ARTIFACTS_HINT = None            # e.g. "birdclef-exp015c-v18-artifacts"

RUN_TTA_VARIANTS = True
SAVE_RESULTS = True

print({
    'COMPETITION_HINT': COMPETITION_HINT,
    'PERCH_CACHE_HINT': PERCH_CACHE_HINT,
    'ARTIFACTS_HINT': ARTIFACTS_HINT,
    'RUN_TTA_VARIANTS': RUN_TTA_VARIANTS,
    'SAVE_RESULTS': SAVE_RESULTS,
})


In [ ]:
# Cell 1 — Imports, path resolution, and runtime setup
import gc
from contextlib import nullcontext
import json
import pickle
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

_WALL_START = time.time()
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
OUTPUT_DIR = Path('/kaggle/working/exp_019_v18_postproc_ablation') if Path('/kaggle/working').exists() else Path('experiments/outputs/exp_019_v18_postproc_ablation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
AMP_ENABLED = DEVICE.type == 'cuda'

def autocast_context():
    if AMP_ENABLED:
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()

def to_device_tensor(x, dtype):
    return torch.as_tensor(x, dtype=dtype, device=DEVICE)

def tensor_to_numpy(x):
    return x.detach().cpu().numpy()

print('Benchmark device:', DEVICE)
print('Output dir:', OUTPUT_DIR)

def resolve_competition_dir():
    candidates = [
        Path('/kaggle/input/competitions/birdclef-2026'),
        Path('/kaggle/input/birdclef-2026'),
    ]
    for p in candidates:
        if (p / 'sample_submission.csv').exists() and (p / 'taxonomy.csv').exists():
            return p
    found = []
    for p in INPUT_ROOT.rglob('sample_submission.csv'):
        parent = p.parent
        if (parent / 'taxonomy.csv').exists():
            if COMPETITION_HINT is None or COMPETITION_HINT.lower() in str(parent).lower():
                found.append(parent)
    if not found:
        raise FileNotFoundError('Could not resolve BirdCLEF competition directory')
    return sorted(found)[0]

def resolve_artifacts_dir():
    found = []
    for p in INPUT_ROOT.rglob('artifacts_manifest.json'):
        parent = p.parent
        if ARTIFACTS_HINT is None or ARTIFACTS_HINT.lower() in str(parent).lower():
            found.append(parent)
    if not found:
        raise FileNotFoundError('Could not resolve exp_015d artifact dataset under input roots')
    return sorted(found)[0]

def resolve_full_cache_paths():
    candidates = []
    for p in INPUT_ROOT.rglob('full_perch_meta.parquet'):
        parent = p.parent
        npz = parent / 'full_perch_arrays.npz'
        if npz.exists() and (PERCH_CACHE_HINT is None or PERCH_CACHE_HINT.lower() in str(parent).lower()):
            candidates.append((p, npz))
    if not candidates:
        raise FileNotFoundError('Could not resolve full Perch cache dataset under input roots')
    return sorted(candidates)[0]

BASE = resolve_competition_dir()
ARTIFACTS_DIR = resolve_artifacts_dir()
CACHE_META, CACHE_NPZ = resolve_full_cache_paths()

print('Competition dir:', BASE)
print('Artifacts dir:', ARTIFACTS_DIR)
print('Full cache meta:', CACHE_META)
print('Full cache npz:', CACHE_NPZ)


In [ ]:
# Cell 2 — Load competition truth and cached full-file Perch outputs
SR = 32000
WINDOW_SEC = 5
N_WINDOWS = 12

sample_sub = pd.read_csv(BASE / 'sample_submission.csv')
taxonomy = pd.read_csv(BASE / 'taxonomy.csv')
soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')
meta_full = pd.read_parquet(CACHE_META)
arr = np.load(CACHE_NPZ)
scores_full_raw = arr['scores_full_raw'].astype(np.float32)
emb_full = arr['emb_full'].astype(np.float32)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

soundscape_labels['primary_label'] = soundscape_labels['primary_label'].astype(str)
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)

def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(';') if t.strip()]

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))

FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'site': None, 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean['label_list']):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)
full_truth_aligned = full_truth.set_index('row_id').loc[meta_full['row_id']].reset_index()
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()].astype(np.float32)

assert np.all(full_truth_aligned['filename'].values == meta_full['filename'].values)
assert np.all(full_truth_aligned['row_id'].values == meta_full['row_id'].values)
assert len(meta_full) == scores_full_raw.shape[0] == emb_full.shape[0] == Y_FULL.shape[0]

CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()
TEXTURE_TAXA = {'Amphibia', 'Insecta'}

print('meta_full:', meta_full.shape)
print('scores_full_raw:', scores_full_raw.shape)
print('emb_full:', emb_full.shape)
print('Y_FULL:', Y_FULL.shape)
print('Trusted full files:', len(full_files))
print('Active classes:', int((Y_FULL.sum(axis=0) > 0).sum()))


In [ ]:
# Cell 3 — Helper utilities

def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    if keep.sum() == 0:
        return float('nan')
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')


def seq_features_1d(v):
    assert len(v) % N_WINDOWS == 0, 'Expected full-file blocks of 12 windows'
    x = v.reshape(-1, N_WINDOWS)
    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v = np.repeat(x.max(axis=1), N_WINDOWS)
    std_v = np.repeat(x.std(axis=1), N_WINDOWS)
    return prev_v, next_v, mean_v, max_v, std_v


def build_class_features(emb_proj, raw_col, prior_col, base_col):
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)
    diff_mean = base_col - mean_base
    diff_prev = base_col - prev_base
    diff_next = base_col - next_base
    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        std_base[:, None],
        diff_mean[:, None],
        diff_prev[:, None],
        diff_next[:, None],
        (raw_col * prior_col)[:, None],
        (raw_col * base_col)[:, None],
        (prior_col * base_col)[:, None],
    ], axis=1)
    return feats.astype(np.float32, copy=False)


def reshape_to_files(flat_array, meta_df, n_windows=N_WINDOWS):
    filenames = meta_df['filename'].to_numpy()
    unique_files = []
    seen = set()
    for f in filenames:
        if f not in seen:
            unique_files.append(f)
            seen.add(f)
    n_files = len(unique_files)
    assert len(flat_array) == n_files * n_windows, f'Expected {n_files * n_windows} rows, got {len(flat_array)}'
    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files


def get_file_metadata(meta_df, file_list, site_to_idx, n_sites_max):
    file_to_row = {}
    filenames = meta_df['filename'].to_numpy()
    sites = meta_df['site'].to_numpy()
    hours = meta_df['hour_utc'].to_numpy()
    for i, f in enumerate(filenames):
        if f not in file_to_row:
            file_to_row[f] = i
    site_ids = np.zeros(len(file_list), dtype=np.int64)
    hour_ids = np.zeros(len(file_list), dtype=np.int64)
    for fi, fname in enumerate(file_list):
        row = file_to_row.get(fname)
        if row is not None:
            sid = site_to_idx.get(str(sites[row]), 0)
            site_ids[fi] = min(sid, n_sites_max - 1)
            hour_ids[fi] = int(hours[row]) % 24
    return site_ids, hour_ids


def optimize_per_class_thresholds(oof_scores, y_true, n_windows=12, thresholds=(0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70)):
    n_classes = oof_scores.shape[1]
    best_thresholds = np.zeros(n_classes, dtype=np.float32)
    best_scores = np.zeros(n_classes, dtype=np.float32)
    for c in range(n_classes):
        y_c = y_true[:, c]
        scores_c = oof_scores[:, c]
        if y_c.sum() == 0:
            best_thresholds[c] = 0.5
            continue
        best_f1, best_t = 0.0, 0.5
        for t in thresholds:
            pred = (scores_c >= t).astype(np.int32)
            tp = ((pred == 1) & (y_c == 1)).sum()
            fp = ((pred == 1) & (y_c == 0)).sum()
            fn = ((pred == 0) & (y_c == 1)).sum()
            prec = tp / (tp + fp + 1e-8)
            rec = tp / (tp + fn + 1e-8)
            f1 = 2 * prec * rec / (prec + rec + 1e-8)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds[c] = best_t
        best_scores[c] = best_f1
    return best_thresholds, best_scores


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

print('Helper utilities loaded.')


In [ ]:
# Cell 4 — Prior fusion helpers
BEST = {
    'lambda_event': 0.45,
    'lambda_texture': 1.1,
    'lambda_proxy_texture': 0.9,
    'smooth_texture': 0.35,
    'smooth_event': 0.15,
}

ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]
idx_active_texture = np.array([label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA], dtype=np.int32)
idx_active_event = np.array([label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA], dtype=np.int32)
idx_mapped_active_texture = idx_active_texture
idx_mapped_active_event = idx_active_event
idx_selected_proxy_active_texture = np.array([], dtype=np.int32)
idx_selected_prioronly_active_texture = np.array([], dtype=np.int32)
idx_selected_prioronly_active_event = np.array([], dtype=np.int32)
idx_unmapped_inactive = np.array([], dtype=np.int32)


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables['global_p'][None, :], n, axis=0).astype(np.float32, copy=True)
    site_idx = np.fromiter((tables['site_to_i'].get(str(s), -1) for s in sites), dtype=np.int32, count=n)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(h), -1) if int(h) >= 0 else -1 for h in hours), dtype=np.int32, count=n)
    sh_idx = np.fromiter((tables['sh_to_i'].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)), dtype=np.int32, count=n)

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def smooth_cols_fixed12(scores, cols, alpha=0.35):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s


def smooth_events_fixed12(scores, cols, alpha=0.15):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev_x, next_x))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s


def fuse_scores_with_tables(base_scores, sites, hours, tables,
                            lambda_event=BEST['lambda_event'],
                            lambda_texture=BEST['lambda_texture'],
                            lambda_proxy_texture=BEST['lambda_proxy_texture'],
                            smooth_texture=BEST['smooth_texture'],
                            smooth_event=BEST['smooth_event']):
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += lambda_event * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += lambda_texture * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += lambda_proxy_texture * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = prior[:, idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = prior[:, idx_unmapped_inactive]

    scores = smooth_cols_fixed12(scores, idx_mapped_active_texture, alpha=smooth_texture)
    scores = smooth_events_fixed12(scores, idx_mapped_active_event, alpha=smooth_event)
    return scores.astype(np.float32), prior.astype(np.float32)

print('Prior helpers loaded.')


In [ ]:
# Cell 5 — Model classes used by the fixed V18 artifact stack
class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        A = A.unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_size, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)
        h = torch.zeros(B_size, D, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            dt_t = dt[:, t, :]
            dA = torch.exp(A[None, :, :] * dt_t[:, :, None])
            dB = dt_t[:, :, None] * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            y_t = (h * C[:, t, None, :]).sum(-1)
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]

class TemporalCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        attn_out, _ = self.attn(x, x, x)
        x = residual + attn_out
        residual = x
        x = self.norm2(x)
        x = residual + self.ffn(x)
        return x

class ProtoSSMv2(nn.Module):
    def __init__(self, d_input=1536, d_model=192, d_state=16,
                 n_ssm_layers=2, n_classes=234, n_windows=12,
                 dropout=0.2, n_sites=20, meta_dim=16,
                 use_cross_attn=True, cross_attn_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.ssm_fwd = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_bwd = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(n_ssm_layers)])
        self.ssm_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.ssm_drop = nn.ModuleList([nn.Dropout(dropout) for _ in range(n_ssm_layers)])
        self.use_cross_attn = use_cross_attn
        if use_cross_attn:
            self.cross_attn = TemporalCrossAttention(d_model, n_heads=cross_attn_heads, dropout=dropout)
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))
        self.n_families = 0
        self.family_head = None

    def init_prototypes_from_data(self, embeddings, labels):
        with torch.no_grad():
            h = self.input_proj(embeddings)
            for c in range(self.n_classes):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def init_family_head(self, n_families, class_to_family):
        self.n_families = n_families
        self.family_head = nn.Linear(self.d_model, n_families)
        self.register_buffer('class_to_family', torch.tensor(class_to_family, dtype=torch.long))

    def forward(self, emb, logits, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb)
        if site_ids is not None and hours is not None:
            site_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            hour_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([site_e, hour_e], dim=-1))
            h = h + meta.unsqueeze(1)
        h = h + self.pos_enc[:, :T, :]
        for ssm_f, ssm_b, merge, norm, drop in zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm, self.ssm_drop):
            residual = h
            h_f = ssm_f(h)
            h_b = ssm_b(h.flip(1)).flip(1)
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = drop(h)
            h = norm(h + residual)
        if self.use_cross_attn:
            h = self.cross_attn(h)
        proto = F.normalize(self.prototypes, dim=-1)
        h_norm = F.normalize(h, dim=-1)
        proto_logits = torch.einsum('btd,cd->btc', h_norm, proto) * F.softplus(self.proto_temp)
        alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
        fused = alpha * proto_logits + (1.0 - alpha) * logits + self.class_bias[None, None, :]
        family_out = self.family_head(h.mean(dim=1)) if self.family_head is not None else None
        return fused, family_out, h

class ResidualSSM(nn.Module):
    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                 n_classes=234, n_windows=12, dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm = nn.LayerNorm(d_model)
        self.ssm_drop = nn.Dropout(dropout)
        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass_scores, site_ids=None, hours=None):
        x = torch.cat([emb, first_pass_scores], dim=-1)
        h = self.input_proj(x)
        if site_ids is not None and hours is not None:
            site_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            hour_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([site_e, hour_e], dim=-1))
            h = h + meta.unsqueeze(1)
        h = h + self.pos_enc[:, :h.shape[1], :]
        residual = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.ssm_merge(torch.cat([h_f, h_b], dim=-1))
        h = self.ssm_drop(h)
        h = self.ssm_norm(h + residual)
        return self.output_head(h)

print('Model classes loaded.')


In [ ]:
# Cell 6 — Load fixed exp_015d artifacts
manifest = json.loads((ARTIFACTS_DIR / 'artifacts_manifest.json').read_text())
with open(ARTIFACTS_DIR / manifest['artifact_files']['prior_tables'], 'rb') as f:
    final_prior_tables = pickle.load(f)
with open(ARTIFACTS_DIR / manifest['artifact_files']['sklearn'], 'rb') as f:
    sk_artifacts = pickle.load(f)

emb_scaler = sk_artifacts['emb_scaler']
emb_pca = sk_artifacts['emb_pca']
probe_models = sk_artifacts['probe_models']
BEST_PROBE = manifest['best_probe']
ENSEMBLE_WEIGHT_PROTO = float(manifest['ensemble_weight_proto'])
CORRECTION_WEIGHT = float(manifest.get('correction_weight', 0.0))
site_to_idx = {str(k): int(v) for k, v in manifest['site_to_idx'].items()}
class_to_family = [int(x) for x in manifest['class_to_family']]
fam_to_idx = {str(k): int(v) for k, v in manifest['fam_to_idx'].items()}
n_families = int(manifest['n_families'])
n_sites_cfg = int(manifest['n_sites_cfg'])
CFG = {
    'proxy_reduce': manifest['proxy_reduce'],
    'temperature': manifest['temperature'],
    'file_level_top_k': int(manifest.get('file_level_top_k', 0)),
    'tta_shifts': [int(x) for x in manifest.get('tta_shifts', [0])],
    'rank_aware_scale': bool(manifest.get('rank_aware_scale', False)),
    'rank_aware_power': float(manifest.get('rank_aware_power', 0.5)),
    'delta_shift_alpha': float(manifest.get('delta_shift_alpha', 0.0)),
    'proto_ssm': manifest.get('proto_ssm', {}),
    'residual_ssm': manifest.get('residual_ssm', {}),
}

threshold_rel = manifest['artifact_files'].get('thresholds', 'per_class_thresholds.npy')
PER_CLASS_THRESHOLDS = np.load(ARTIFACTS_DIR / threshold_rel).astype(np.float32)

calibrators_rel = manifest['artifact_files'].get('calibrators')
CALIBRATORS = None
if calibrators_rel:
    cal_path = ARTIFACTS_DIR / calibrators_rel
    if cal_path.exists():
        with open(cal_path, 'rb') as f:
            CALIBRATORS = pickle.load(f)

proto_ckpt = torch.load(ARTIFACTS_DIR / manifest['artifact_files']['proto'], map_location=DEVICE)
proto_cfg = proto_ckpt['proto_ssm']
model = ProtoSSMv2(
    d_input=int(emb_pca.mean_.shape[0]) if hasattr(emb_pca, 'mean_') else emb_full.shape[1],
    d_model=proto_cfg['d_model'],
    d_state=proto_cfg['d_state'],
    n_ssm_layers=proto_cfg['n_ssm_layers'],
    n_classes=int(manifest['n_classes']),
    n_windows=int(manifest['n_windows']),
    dropout=proto_cfg['dropout'],
    n_sites=proto_cfg['n_sites'],
    meta_dim=proto_cfg['meta_dim'],
    use_cross_attn=proto_cfg.get('use_cross_attn', True),
    cross_attn_heads=proto_cfg.get('cross_attn_heads', 4),
).to(DEVICE)
model.init_family_head(n_families, class_to_family)
model.load_state_dict(proto_ckpt['state_dict'], strict=True)
model.eval()

res_model = None
if manifest.get('has_residual', False) and manifest['artifact_files'].get('residual'):
    res_ckpt = torch.load(ARTIFACTS_DIR / manifest['artifact_files']['residual'], map_location=DEVICE)
    res_cfg = res_ckpt['residual_ssm']
    res_model = ResidualSSM(
        d_input=int(emb_pca.mean_.shape[0]) if hasattr(emb_pca, 'mean_') else emb_full.shape[1],
        d_scores=int(manifest['n_classes']),
        d_model=res_cfg['d_model'],
        d_state=res_cfg['d_state'],
        n_classes=int(manifest['n_classes']),
        n_windows=int(manifest['n_windows']),
        dropout=res_cfg['dropout'],
        n_sites=proto_cfg['n_sites'],
        meta_dim=8,
    ).to(DEVICE)
    res_model.load_state_dict(res_ckpt['state_dict'], strict=True)
    res_model.eval()

print('Loaded fixed exp_015d artifacts.')
print('  Proto params:', manifest['proto_parameters'])
print('  Residual enabled:', res_model is not None)
print('  Probe models:', len(probe_models))
print('  Threshold mean:', float(PER_CLASS_THRESHOLDS.mean()))
print('  Calibrators present:', CALIBRATORS is not None)


In [ ]:
# Cell 7 — Replay fixed V18 stack on cached full rows
emb_files, file_list = reshape_to_files(emb_full, meta_full)
logits_files, _ = reshape_to_files(scores_full_raw, meta_full)
site_ids_all, hours_all = get_file_metadata(meta_full, file_list, site_to_idx, n_sites_cfg)

BATCH_FILES = 256 if DEVICE.type == 'cuda' else 64

base_scores, prior_scores = fuse_scores_with_tables(
    scores_full_raw,
    sites=meta_full['site'].to_numpy(),
    hours=meta_full['hour_utc'].to_numpy(),
    tables=final_prior_tables,
)

emb_scaled = emb_scaler.transform(emb_full)
Z_FULL = emb_pca.transform(emb_scaled).astype(np.float32)
del emb_scaled

mlp_t0 = time.time()
mlp_scores = base_scores.copy()
for cls_idx, clf in probe_models.items():
    X_cls = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=prior_scores[:, cls_idx],
        base_col=base_scores[:, cls_idx],
    )
    if hasattr(clf, 'predict_proba'):
        prob = clf.predict_proba(X_cls)[:, 1].astype(np.float32)
        pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
    else:
        pred = clf.decision_function(X_cls).astype(np.float32)
    alpha = float(BEST_PROBE['alpha'])
    mlp_scores[:, cls_idx] = (1.0 - alpha) * base_scores[:, cls_idx] + alpha * pred
print(f'MLP replay time: {time.time() - mlp_t0:.1f}s')

PROTO_CACHE = {}
STACK_CACHE = {}

def proto_forward_batched(model, emb_files, logits_files, site_ids_all, hours_all, batch_files):
    outputs = []
    with torch.no_grad():
        for start in range(0, len(emb_files), batch_files):
            end = min(start + batch_files, len(emb_files))
            emb_tensor = to_device_tensor(emb_files[start:end], torch.float32)
            logits_tensor = to_device_tensor(logits_files[start:end], torch.float32)
            site_tensor = to_device_tensor(site_ids_all[start:end], torch.long)
            hour_tensor = to_device_tensor(hours_all[start:end], torch.long)
            with autocast_context():
                proto_out, _, _ = model(emb_tensor, logits_tensor, site_ids=site_tensor, hours=hour_tensor)
            outputs.append(tensor_to_numpy(proto_out))
            del emb_tensor, logits_tensor, site_tensor, hour_tensor, proto_out
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
    return np.concatenate(outputs, axis=0)


def temporal_shift_tta_batched(emb_files, logits_files, model, site_ids, hours, shifts=(0, 1, -1), batch_files=64):
    all_preds = []
    for shift in shifts:
        if shift == 0:
            e = emb_files
            l = logits_files
        else:
            e = np.roll(emb_files, shift, axis=1)
            l = np.roll(logits_files, shift, axis=1)
        pred = proto_forward_batched(model, e, l, site_ids, hours, batch_files)
        if shift != 0:
            pred = np.roll(pred, -shift, axis=1)
        all_preds.append(pred)
    return np.mean(all_preds, axis=0)


def residual_forward_batched(res_model, emb_files, first_pass_files, site_ids_all, hours_all, batch_files):
    outputs = []
    with torch.no_grad():
        for start in range(0, len(emb_files), batch_files):
            end = min(start + batch_files, len(emb_files))
            emb_tensor = to_device_tensor(emb_files[start:end], torch.float32)
            first_pass_tensor = to_device_tensor(first_pass_files[start:end], torch.float32)
            site_tensor = to_device_tensor(site_ids_all[start:end], torch.long)
            hour_tensor = to_device_tensor(hours_all[start:end], torch.long)
            with autocast_context():
                correction = res_model(emb_tensor, first_pass_tensor, site_ids=site_tensor, hours=hour_tensor)
            outputs.append(tensor_to_numpy(correction))
            del emb_tensor, first_pass_tensor, site_tensor, hour_tensor, correction
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
    return np.concatenate(outputs, axis=0)


def replay_v18_stack(tta_shifts=(0,)):
    key = tuple(int(x) for x in tta_shifts)
    if key in STACK_CACHE:
        return STACK_CACHE[key]

    t0 = time.time()
    if key == (0,):
        proto_scores_flat = proto_forward_batched(
            model, emb_files, logits_files, site_ids_all, hours_all, BATCH_FILES
        ).reshape(-1, N_CLASSES).astype(np.float32)
    else:
        proto_scores_flat = temporal_shift_tta_batched(
            emb_files, logits_files, model, site_ids_all, hours_all, shifts=key, batch_files=BATCH_FILES
        ).reshape(-1, N_CLASSES).astype(np.float32)

    final_scores = (
        ENSEMBLE_WEIGHT_PROTO * proto_scores_flat +
        (1.0 - ENSEMBLE_WEIGHT_PROTO) * mlp_scores
    ).astype(np.float32)

    if res_model is not None and CORRECTION_WEIGHT > 0:
        first_pass_files, _ = reshape_to_files(final_scores, meta_full)
        correction_flat = residual_forward_batched(
            res_model, emb_files, first_pass_files, site_ids_all, hours_all, BATCH_FILES
        ).reshape(-1, N_CLASSES).astype(np.float32)
        final_scores = final_scores + CORRECTION_WEIGHT * correction_flat
    result = {
        'tta_shifts': key,
        'proto_scores_flat': proto_scores_flat,
        'final_scores': final_scores.astype(np.float32),
        'wall_time': float(time.time() - t0),
    }
    STACK_CACHE[key] = result
    return result

baseline_replay = replay_v18_stack(tuple(CFG.get('tta_shifts', [0])))
proto_scores_flat = baseline_replay['proto_scores_flat']
final_scores = baseline_replay['final_scores']
print('Replayed fixed V18 stack.')
print('  tta_shifts:', baseline_replay['tta_shifts'])
print('  replay wall time:', round(baseline_replay['wall_time'], 3), 's')
print('  proto_scores_flat:', proto_scores_flat.shape)
print('  final_scores:', final_scores.shape)
print('  macro_auc(logit-like final scores):', float(macro_auc_skip_empty(Y_FULL, final_scores)))


In [ ]:
# Cell 8 — Run low-risk postprocess ablation sweep

def file_level_confidence_scale(preds, n_windows=12, top_k=2):
    N, C = preds.shape
    assert N % n_windows == 0
    view = preds.reshape(-1, n_windows, C)
    sorted_view = np.sort(view, axis=1)
    top_k_mean = sorted_view[:, -top_k:, :].mean(axis=1, keepdims=True)
    scaled = view * top_k_mean
    return np.clip(scaled.reshape(N, C), 0.0, 1.0)


def rank_aware_scaling(scores, n_windows=12, power=0.5):
    N, C = scores.shape
    assert N % n_windows == 0
    view = scores.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)
    scale = np.power(np.clip(file_max, 0.0, 1.0), power)
    scaled = view * scale
    return np.clip(scaled.reshape(N, C), 0.0, 1.0)


def adaptive_delta_smooth(scores, n_windows=12, base_alpha=0.20):
    result = scores.copy().reshape(-1, n_windows, scores.shape[1])
    original = scores.reshape(-1, n_windows, scores.shape[1])
    for i in range(1, n_windows - 1):
        conf = original[:, i, :].max(axis=-1, keepdims=True)
        a = base_alpha * (1.0 - conf)
        neighbor_avg = (original[:, i - 1, :] + original[:, i + 1, :]) / 2.0
        result[:, i, :] = (1.0 - a) * original[:, i, :] + a * neighbor_avg
    return np.clip(result.reshape(scores.shape), 0.0, 1.0)


def apply_per_class_thresholds(scores, thresholds):
    N, C = scores.shape
    assert C == len(thresholds)
    scaled = np.copy(scores)
    for c in range(C):
        t = float(thresholds[c])
        mask_above = scores[:, c] > t
        scaled[mask_above, c] = 0.5 + 0.5 * (scores[mask_above, c] - t) / (1 - t + 1e-8)
        scaled[~mask_above, c] = 0.5 * scores[~mask_above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)


def apply_file_level_calibrators(scores, calibrators, n_windows=12):
    scores_files = scores.reshape(-1, n_windows, scores.shape[1]).copy()
    file_max = scores_files.max(axis=1)
    calibrated_max = file_max.copy()
    for ci, cal in enumerate(calibrators):
        if cal is None:
            continue
        calibrated_max[:, ci] = np.clip(cal.transform(file_max[:, ci]), 0.0, 1.0)
    scale = calibrated_max / np.clip(file_max, 1e-6, None)
    scale = np.clip(scale, 0.25, 4.0)
    scores_files *= scale[:, None, :]
    return np.clip(scores_files.reshape(scores.shape), 0.0, 1.0)


def build_tempered_probs(logit_scores):
    class_temperatures = np.ones(N_CLASSES, dtype=np.float32) * float(CFG['temperature']['aves'])
    for ci, label in enumerate(PRIMARY_LABELS):
        if CLASS_NAME_MAP.get(label, 'Aves') in TEXTURE_TAXA:
            class_temperatures[ci] = float(CFG['temperature']['texture'])
    return sigmoid(logit_scores / class_temperatures[None, :]).astype(np.float32)


def macro_auc_subset(y_true, y_score, idx):
    idx = np.array(idx, dtype=np.int64)
    idx = idx[(idx >= 0) & (idx < y_true.shape[1])]
    if len(idx) == 0:
        return float('nan')
    keep = y_true[:, idx].sum(axis=0) > 0
    if keep.sum() == 0:
        return float('nan')
    return roc_auc_score(y_true[:, idx][:, keep], y_score[:, idx][:, keep], average='macro')


def evaluate_variant(name, replay, top_k, rank_aware, rank_power, delta_alpha, use_thresholds, aves_smooth_alpha=0.0, use_calibration=False):
    probs = build_tempered_probs(replay['final_scores'])
    if use_calibration and CALIBRATORS is not None:
        probs = apply_file_level_calibrators(probs, CALIBRATORS, n_windows=N_WINDOWS)
    if top_k > 0:
        probs = file_level_confidence_scale(probs, n_windows=N_WINDOWS, top_k=top_k)
    if rank_aware:
        probs = rank_aware_scaling(probs, n_windows=N_WINDOWS, power=rank_power)
    if aves_smooth_alpha > 0:
        probs = smooth_cols_fixed12(probs, idx_mapped_active_event, alpha=aves_smooth_alpha)
    if delta_alpha > 0:
        probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=delta_alpha)
    if use_thresholds:
        probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)

    file_probs = probs.reshape(-1, N_WINDOWS, N_CLASSES).max(axis=1)
    file_y = Y_FULL.reshape(-1, N_WINDOWS, N_CLASSES).max(axis=1)

    result = {
        'variant': name,
        'tta_shifts': list(replay['tta_shifts']),
        'replay_wall_time': float(replay['wall_time']),
        'use_calibration': bool(use_calibration and CALIBRATORS is not None),
        'top_k': int(top_k),
        'rank_aware': bool(rank_aware),
        'rank_power': float(rank_power),
        'delta_alpha': float(delta_alpha),
        'aves_smooth_alpha': float(aves_smooth_alpha),
        'use_thresholds': bool(use_thresholds),
        'row_macro_auc': float(macro_auc_skip_empty(Y_FULL, probs)),
        'file_macro_auc': float(macro_auc_skip_empty(file_y, file_probs)),
        'row_texture_auc': float(macro_auc_subset(Y_FULL, probs, idx_mapped_active_texture)),
        'file_texture_auc': float(macro_auc_subset(file_y, file_probs, idx_mapped_active_texture)),
        'row_event_auc': float(macro_auc_subset(Y_FULL, probs, idx_mapped_active_event)),
        'file_event_auc': float(macro_auc_subset(file_y, file_probs, idx_mapped_active_event)),
        'prob_mean': float(probs.mean()),
        'prob_max': float(probs.max()),
    }
    return result

manifest_tta = tuple(CFG.get('tta_shifts', [0]))
variant_specs = [
    {'name': 'manifest_baseline', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'no_thresholds', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': False, 'use_calibration': CALIBRATORS is not None},
    {'name': 'no_file_scale', 'tta_shifts': manifest_tta, 'top_k': 0, 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'no_rank_aware', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': False, 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'no_delta_smooth', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': 0.0, 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'topk3_manifest', 'tta_shifts': manifest_tta, 'top_k': 3, 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'rank035_manifest', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': True, 'rank_power': 0.35, 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'rank045_manifest', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': True, 'rank_power': 0.45, 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'delta010_manifest', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': 0.10, 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    {'name': 'aves_smooth005_manifest', 'tta_shifts': manifest_tta, 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.05, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
]

if RUN_TTA_VARIANTS:
    variant_specs.extend([
        {'name': 'tta_shift_p1', 'tta_shifts': (0, 1), 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
        {'name': 'tta_shift_pm1', 'tta_shifts': (0, 1, -1), 'top_k': CFG.get('file_level_top_k', 0), 'rank_aware': CFG.get('rank_aware_scale', False), 'rank_power': CFG.get('rank_aware_power', 0.5), 'delta_alpha': CFG.get('delta_shift_alpha', 0.0), 'aves_smooth_alpha': 0.0, 'use_thresholds': True, 'use_calibration': CALIBRATORS is not None},
    ])

results = []
for spec in variant_specs:
    replay = replay_v18_stack(spec['tta_shifts'])
    result = evaluate_variant(
        name=spec['name'],
        replay=replay,
        top_k=spec['top_k'],
        rank_aware=spec['rank_aware'],
        rank_power=spec['rank_power'],
        delta_alpha=spec['delta_alpha'],
        use_thresholds=spec['use_thresholds'],
        aves_smooth_alpha=spec['aves_smooth_alpha'],
        use_calibration=spec['use_calibration'],
    )
    results.append(result)
    print(f"{result['variant']}: row_auc={result['row_macro_auc']:.6f}, file_auc={result['file_macro_auc']:.6f}, texture_file_auc={result['file_texture_auc']:.6f}")

variant_results = pd.DataFrame(results).sort_values(['row_macro_auc', 'file_macro_auc'], ascending=False).reset_index(drop=True)
display(variant_results)

best_row = variant_results.iloc[0].to_dict()
baseline_row = variant_results.loc[variant_results['variant'] == 'manifest_baseline'].iloc[0].to_dict()
report_snapshot = {
    'experiment_id': 'exp_019',
    'experiment_name': 'v18_postproc_ablation',
    'artifacts_dir': str(ARTIFACTS_DIR),
    'cache_meta': str(CACHE_META),
    'device': str(DEVICE),
    'n_variants': int(len(variant_results)),
    'baseline_variant': baseline_row['variant'],
    'baseline_row_macro_auc': float(baseline_row['row_macro_auc']),
    'best_variant': best_row['variant'],
    'best_row_macro_auc': float(best_row['row_macro_auc']),
    'delta_vs_baseline': float(best_row['row_macro_auc'] - baseline_row['row_macro_auc']),
}
print(report_snapshot)

if SAVE_RESULTS:
    variant_results.to_csv(OUTPUT_DIR / 'variant_results.csv', index=False)
    (OUTPUT_DIR / 'report_snapshot.json').write_text(json.dumps(report_snapshot, indent=2))
    print('Saved results to:', OUTPUT_DIR)
